# Inspeção Técnica do `all-MiniLM-L12-v2`

Este notebook realiza uma inspeção técnica completa do modelo **sentence-transformers/all-MiniLM-L12-v2**,
cobrindo arquitetura, tokenizer, pooling, normalização, truncamento e exemplos de embeddings.

> **Objetivo:** Documentar as características internas do modelo antes de executar a avaliação formal,
> garantindo compreensão plena de seus parâmetros e comportamentos.

---

## Seção 1 — Setup e Importações

Importamos as bibliotecas necessárias e verificamos o dispositivo disponível (CPU ou GPU).

In [ ]:
# ── Importações ───────────────────────────────────────────────────────────────
import json
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import matplotlib
matplotlib.use('Agg')  # backend não-interativo para salvar figuras
import matplotlib.pyplot as plt

try:
    from transformers import AutoModel, AutoTokenizer
    from sentence_transformers import SentenceTransformer
except ImportError as e:
    raise ImportError(
        f"Dependência ausente: {e}. "
        "Instale com: pip install transformers sentence-transformers"
    )

warnings.filterwarnings('ignore')

# ── Configurações globais ─────────────────────────────────────────────────────
MODEL_NAME = "sentence-transformers/all-MiniLM-L12-v2"
device = "cuda" if torch.cuda.is_available() else "cpu"

print(f"Dispositivo  : {device}")
print(f"PyTorch      : {torch.__version__}")
print(f"NumPy        : {np.__version__}")
print(f"Pandas       : {pd.__version__}")
print(f"Modelo alvo  : {MODEL_NAME}")

---
## Seção 2 — Carregamento do Modelo

Carregamos o modelo de duas formas complementares:
1. **SentenceTransformer** — interface de alto nível, já inclui pooling e normalização.
2. **AutoModel + AutoTokenizer** — acesso direto à arquitetura Transformer base para inspeção interna.

In [ ]:
# ── Carrega via SentenceTransformer ──────────────────────────────────────────
print("Carregando SentenceTransformer...")
try:
    st_model = SentenceTransformer(MODEL_NAME, device=device)
    print("  SentenceTransformer carregado com sucesso.")
except Exception as e:
    raise RuntimeError(f"Erro ao carregar SentenceTransformer: {e}")

# Dimensão dos embeddings gerados
embedding_dim = st_model.get_sentence_embedding_dimension()
print(f"\n  Dimensão dos embeddings : {embedding_dim}  (esperado: 384)")
print(f"  max_seq_length          : {st_model.max_seq_length}")

# similarity_fn_name pode não existir em versões antigas da biblioteca
if hasattr(st_model, 'similarity_fn_name'):
    print(f"  similarity_fn_name      : {st_model.similarity_fn_name}")
else:
    print("  similarity_fn_name      : (atributo não disponível nesta versão)")

# ── Carrega via HuggingFace Transformers ─────────────────────────────────────
print("\nCarregando AutoTokenizer e AutoModel...")
try:
    hf_tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
    hf_model = AutoModel.from_pretrained(MODEL_NAME).to(device)
    hf_model.eval()  # modo de inferência
    print("  AutoModel e AutoTokenizer carregados com sucesso.")
except Exception as e:
    raise RuntimeError(f"Erro ao carregar via HuggingFace: {e}")

---
## Seção 3 — Arquitetura e Parâmetros

Inspecionamos a configuração do modelo base: número de camadas, dimensões, cabeças de atenção,
vocabulário e contagem total de parâmetros. Os resultados são salvos em JSON para rastreabilidade.

In [ ]:
# ── Extrai o modelo base do SentenceTransformer ───────────────────────────────
# O primeiro módulo do SentenceTransformer é o Transformer base
try:
    base_module = st_model[0]           # objeto Transformer (sentence_transformers)
    base_model  = base_module.auto_model  # nn.Module subjacente
    config      = base_model.config
except (IndexError, AttributeError) as e:
    # Fallback: usa o modelo HuggingFace diretamente
    base_model = hf_model
    config     = hf_model.config
    print(f"Aviso: usando hf_model como fallback ({e})")

print("=" * 60)
print("CONFIGURAÇÃO DO MODELO BASE")
print("=" * 60)
print(config)

# ── Contagem de parâmetros ────────────────────────────────────────────────────
total_params     = sum(p.numel() for p in hf_model.parameters())
trainable_params = sum(p.numel() for p in hf_model.parameters() if p.requires_grad)

print(f"\nTotal de parâmetros    : {total_params:,}")
print(f"Parâmetros treináveis  : {trainable_params:,}")
print(f"Parâmetros congelados  : {total_params - trainable_params:,}")

# ── Extrai métricas individuais ───────────────────────────────────────────────
num_hidden_layers   = getattr(config, 'num_hidden_layers',   None)
hidden_size         = getattr(config, 'hidden_size',         None)
num_attention_heads = getattr(config, 'num_attention_heads', None)
vocab_size          = getattr(config, 'vocab_size',          None)
max_position_emb    = getattr(config, 'max_position_embeddings', None)
intermediate_size   = getattr(config, 'intermediate_size',   None)

print("\n" + "=" * 60)
print("MÉTRICAS DE ARQUITETURA")
print("=" * 60)
print(f"  num_hidden_layers      : {num_hidden_layers}")
print(f"  hidden_size            : {hidden_size}")
print(f"  num_attention_heads    : {num_attention_heads}")
print(f"  vocab_size             : {vocab_size}")
print(f"  max_position_embeddings: {max_position_emb}")
print(f"  intermediate_size (FF) : {intermediate_size}")

# ── Salva em JSON ─────────────────────────────────────────────────────────────
architecture_dict = {
    "model_name"            : MODEL_NAME,
    "device"                : device,
    "embedding_dimension"   : embedding_dim,
    "num_hidden_layers"     : num_hidden_layers,
    "hidden_size"           : hidden_size,
    "num_attention_heads"   : num_attention_heads,
    "vocab_size"            : vocab_size,
    "max_position_embeddings": max_position_emb,
    "intermediate_size"     : intermediate_size,
    "total_params"          : total_params,
    "trainable_params"      : trainable_params
}

metrics_dir = Path("../results/metrics")
metrics_dir.mkdir(parents=True, exist_ok=True)
arch_path = metrics_dir / "01_architecture.json"

with open(arch_path, "w", encoding="utf-8") as f:
    json.dump(architecture_dict, f, indent=2, ensure_ascii=False)

print(f"\nArquivo salvo em: {arch_path.resolve()}")

---
## Seção 4 — Tokenizer

O tokenizer é responsável por converter texto em IDs numéricos compreensíveis pelo modelo.
Inspecionamos suas configurações principais e demonstramos a tokenização de frases de exemplo.

In [ ]:
# ── Informações gerais do tokenizer ──────────────────────────────────────────
print("=" * 60)
print("INFORMAÇÕES DO TOKENIZER")
print("=" * 60)
print(f"  Tipo               : {type(hf_tokenizer).__name__}")
print(f"  vocab_size         : {hf_tokenizer.vocab_size:,}")
print(f"  model_max_length   : {hf_tokenizer.model_max_length}")
print(f"  Token [CLS]        : '{hf_tokenizer.cls_token}'  (id={hf_tokenizer.cls_token_id})")
print(f"  Token [SEP]        : '{hf_tokenizer.sep_token}'  (id={hf_tokenizer.sep_token_id})")
print(f"  Token [PAD]        : '{hf_tokenizer.pad_token}'  (id={hf_tokenizer.pad_token_id})")
print(f"  Token [UNK]        : '{hf_tokenizer.unk_token}'  (id={hf_tokenizer.unk_token_id})")
print(f"  Token [MASK]       : '{hf_tokenizer.mask_token}'")

# Tokens especiais completos
special_tokens = hf_tokenizer.all_special_tokens
print(f"  Todos tokens especiais ({len(special_tokens)}): {special_tokens}")

# ── Tokenização de frases de exemplo ─────────────────────────────────────────
sample_sentences = [
    "EJA é a Educação de Jovens e Adultos.",
    "O aluno retornou à escola após anos de afastamento.",
    "A alfabetização é um direito fundamental de todos os cidadãos brasileiros.",
    "Qual a diferença entre ensino fundamental e ensino médio na EJA?",
    "Supletivo, EJA e educação continuada são modalidades complementares."
]

print("\n" + "=" * 60)
print("TOKENIZAÇÃO DAS FRASES DE EXEMPLO")
print("=" * 60)

token_records = []
for sent in sample_sentences:
    tokens = hf_tokenizer.tokenize(sent)
    ids    = hf_tokenizer.encode(sent, add_special_tokens=True)
    token_records.append({
        "frase"         : sent[:60] + "..." if len(sent) > 60 else sent,
        "n_tokens"      : len(tokens),
        "n_ids (c/ esp)": len(ids),
        "tokens"        : str(tokens[:10]) + ("..." if len(tokens) > 10 else "")
    })

df_tok = pd.DataFrame(token_records)
print(df_tok.to_string(index=False))

---
## Seção 5 — Pooling e Normalização

O modelo `all-MiniLM-L12-v2` utiliza **mean pooling** sobre os token embeddings da última camada
e, opcionalmente, **normalização L2** para que os vetores tenham norma unitária (essencial para
similaridade de cosseno). Verificamos esse pipeline inspecionando os módulos internos e comparando
o resultado manual com a saída do SentenceTransformer.

In [ ]:
# ── Inspeciona os módulos do SentenceTransformer ──────────────────────────────
print("=" * 60)
print("MÓDULOS DO SentenceTransformer")
print("=" * 60)
for idx, (name, module) in enumerate(st_model.named_modules()):
    # Exibe apenas o primeiro nível (evita saída excessiva)
    if name.count('.') == 0 and name != '':
        print(f"  [{idx:02d}] nome='{name}'  tipo={type(module).__name__}")

# Lista direta dos módulos de pipeline
print("\nMódulos de pipeline (st_model children):")
try:
    modules_list = list(st_model.children())
    for i, mod in enumerate(modules_list):
        print(f"  [{i}] {type(mod).__name__}")
except Exception as e:
    print(f"  Erro ao listar módulos: {e}")

In [ ]:
# ── Demonstração de mean pooling manual ───────────────────────────────────────
frase_teste = "EJA promove a inclusão de adultos na educação formal."

def mean_pooling(model_output, attention_mask):
    """Calcula o mean pooling dos token embeddings ponderado pela attention_mask."""
    token_embeddings = model_output.last_hidden_state           # (1, seq_len, hidden)
    input_mask_exp   = attention_mask.unsqueeze(-1)             # (1, seq_len, 1)
    input_mask_exp   = input_mask_exp.expand_as(token_embeddings).float()
    sum_emb  = torch.sum(token_embeddings * input_mask_exp, dim=1)  # (1, hidden)
    sum_mask = torch.clamp(input_mask_exp.sum(dim=1), min=1e-9)     # (1, hidden)
    return sum_emb / sum_mask

# Tokenização
encoded = hf_tokenizer(
    frase_teste, return_tensors='pt', padding=True, truncation=True
).to(device)

with torch.no_grad():
    output_hf = hf_model(**encoded)

# Mean pooling manual
emb_manual = mean_pooling(output_hf, encoded['attention_mask'])  # (1, 384)

# Normalização L2 manual
emb_manual_norm = torch.nn.functional.normalize(emb_manual, p=2, dim=1)

# Embedding via SentenceTransformer (sem normalização)
emb_st_raw  = st_model.encode(frase_teste, normalize_embeddings=False, convert_to_tensor=True)
# Embedding via SentenceTransformer (com normalização)
emb_st_norm = st_model.encode(frase_teste, normalize_embeddings=True,  convert_to_tensor=True)

# ── Comparação ────────────────────────────────────────────────────────────────
print("=" * 60)
print("COMPARAÇÃO MANUAL vs SentenceTransformer")
print("=" * 60)

# Diferença absoluta máxima entre pooling manual normalizado e ST normalizado
diff = (emb_manual_norm.cpu() - emb_st_norm.cpu()).abs().max().item()
print(f"  Diferença máxima absoluta (manual_norm vs ST_norm) : {diff:.6e}")
print(f"  (Valores próximos de 0 confirmam a equivalência)")

# ── Verificação de norma L2 ───────────────────────────────────────────────────
print("\nVerificação de norma L2:")
norm_manual_raw  = emb_manual.norm(p=2).item()
norm_manual_norm = emb_manual_norm.norm(p=2).item()
norm_st_raw      = emb_st_raw.norm(p=2).item()
norm_st_norm     = emb_st_norm.norm(p=2).item()

print(f"  Manual SEM normalização   : {norm_manual_raw:.6f}")
print(f"  Manual COM normalização   : {norm_manual_norm:.6f}  ← deve ser ≈ 1.0")
print(f"  ST SEM normalização       : {norm_st_raw:.6f}")
print(f"  ST COM normalização       : {norm_st_norm:.6f}  ← deve ser ≈ 1.0")
assert abs(norm_st_norm - 1.0) < 1e-5, "Norma L2 não é 1.0!"
print("\n  ✓ Norma L2 = 1.0 confirmada para normalize_embeddings=True")

---
## Seção 6 — Teste de Truncamento

O modelo possui `max_seq_length = 256` tokens. Textos mais longos são **truncados** automaticamente,
o que pode resultar em perda de informação. Aqui medimos o impacto do truncamento para textos
de tamanhos crescentes.

In [ ]:
# ── Textos de comprimento variável ────────────────────────────────────────────
texts = [
    "Texto curto para teste.",
    " ".join(["palavra"] * 150),
    " ".join(["palavra"] * 300),
    " ".join(["palavra"] * 500)
]

max_seq = st_model.max_seq_length  # geralmente 256

print("=" * 70)
print(f"TESTE DE TRUNCAMENTO  |  max_seq_length = {max_seq}")
print("=" * 70)

truncation_records = []
for i, text in enumerate(texts):
    # Conta tokens SEM truncamento (para saber o tamanho real)
    ids_full = hf_tokenizer.encode(text, add_special_tokens=True, truncation=False)
    n_full   = len(ids_full)

    # Conta tokens COM truncamento no limite do modelo
    ids_trunc = hf_tokenizer.encode(
        text,
        add_special_tokens=True,
        truncation=True,
        max_length=max_seq
    )
    n_trunc = len(ids_trunc)

    tokens_cortados = max(0, n_full - n_trunc)
    foi_truncado    = n_full > max_seq

    pct_perdida = (tokens_cortados / n_full * 100) if n_full > 0 else 0

    rótulo = f"Texto {i+1} ({len(text.split())} palavras)"
    status = "⚠ TRUNCADO" if foi_truncado else "✓ OK"

    print(f"\n  {rótulo}")
    print(f"    Tokens originais    : {n_full}")
    print(f"    Tokens após trunc.  : {n_trunc}")
    print(f"    Tokens cortados     : {tokens_cortados}  ({pct_perdida:.1f}% perdido)")
    print(f"    Status              : {status}")

    truncation_records.append({
        "texto"            : rótulo,
        "tokens_originais"  : n_full,
        "tokens_apos_trunc" : n_trunc,
        "tokens_cortados"   : tokens_cortados,
        "pct_perdida"       : round(pct_perdida, 2),
        "truncado"          : foi_truncado
    })

print("\n" + "-" * 70)
df_trunc = pd.DataFrame(truncation_records)
print("\nTabela resumo:")
print(df_trunc.to_string(index=False))

---
## Seção 7 — Embedding de Exemplo

Codificamos 5 frases relacionadas ao contexto de **EJA (Educação de Jovens e Adultos)**
e analisamos as propriedades dos embeddings gerados: dimensão, estatísticas descritivas
e distribuição dos valores ao longo das 384 dimensões.

In [ ]:
# ── Frases sobre EJA ──────────────────────────────────────────────────────────
eja_sentences = [
    "A EJA oferece oportunidade de conclusão do ensino fundamental para adultos.",
    "Jovens que abandonaram a escola podem retornar pela modalidade EJA.",
    "A certificação pelo ENCCEJA equivale ao diploma de ensino médio.",
    "Educação de adultos combina aprendizado formal com experiências de vida.",
    "Supletivo e EJA são formas de garantir educação inclusiva no Brasil."
]

# Gera embeddings com normalização
embeddings = st_model.encode(
    eja_sentences,
    normalize_embeddings=True,
    show_progress_bar=False,
    batch_size=5
)  # shape: (5, 384)

print("=" * 60)
print("EMBEDDINGS DAS FRASES EJA")
print("=" * 60)
print(f"  Shape dos embeddings : {embeddings.shape}")
print(f"  Dtype                : {embeddings.dtype}")
print(f"  Mínimo global        : {embeddings.min():.6f}")
print(f"  Máximo global        : {embeddings.max():.6f}")
print(f"  Média global         : {embeddings.mean():.6f}")
print(f"  Desvio padrão global : {embeddings.std():.6f}")

# Norma de cada embedding (deve ser 1.0 para todos)
print("\nNorma L2 por frase:")
for i, (sent, emb) in enumerate(zip(eja_sentences, embeddings)):
    norma = np.linalg.norm(emb)
    print(f"  [{i+1}] norma={norma:.6f}  | frase: '{sent[:55]}...'")

# ── Estatísticas por frase ────────────────────────────────────────────────────
stats = []
for i, (sent, emb) in enumerate(zip(eja_sentences, embeddings)):
    stats.append({
        "frase" : f"Frase {i+1}",
        "min"   : round(float(emb.min()), 5),
        "max"   : round(float(emb.max()), 5),
        "média" : round(float(emb.mean()), 5),
        "std"   : round(float(emb.std()), 5),
        "norma" : round(float(np.linalg.norm(emb)), 6)
    })

df_stats = pd.DataFrame(stats)
print("\nEstatísticas descritivas por frase:")
print(df_stats.to_string(index=False))

In [ ]:
# ── Gráfico de linha: valores do embedding por dimensão ───────────────────────
plots_dir = Path("../results/plots")
plots_dir.mkdir(parents=True, exist_ok=True)
plot_path = plots_dir / "01_embeddings_sample.png"

fig, ax = plt.subplots(figsize=(14, 5))

colors = plt.cm.tab10(np.linspace(0, 1, len(eja_sentences)))

for i, (emb, color) in enumerate(zip(embeddings, colors)):
    ax.plot(emb, linewidth=0.7, alpha=0.85, color=color,
            label=f"Frase {i+1}")

ax.axhline(0, color='black', linewidth=0.5, linestyle='--', alpha=0.4)
ax.set_title(
    "Valores dos Embeddings por Dimensão — Frases EJA\n"
    f"(all-MiniLM-L12-v2 | normalize_embeddings=True | dim={embeddings.shape[1]})",
    fontsize=11
)
ax.set_xlabel("Índice da Dimensão (0–383)", fontsize=10)
ax.set_ylabel("Valor do Embedding", fontsize=10)
ax.legend(loc='upper right', fontsize=8, framealpha=0.7)
ax.set_xlim(0, embeddings.shape[1] - 1)
ax.grid(True, alpha=0.25)

plt.tight_layout()
plt.savefig(plot_path, dpi=150, bbox_inches='tight')
plt.close()

print(f"Gráfico salvo em: {plot_path.resolve()}")

---
## Seção 8 — Relatório de Inspeção

Consolidamos todos os achados em um dicionário estruturado, salvamos em JSON e exibimos
uma tabela formatada como resumo final da inspeção técnica.

In [ ]:
import datetime

# ── Monta o relatório completo ────────────────────────────────────────────────
inspection_report = {
    "metadata": {
        "notebook"   : "01_model_inspection.ipynb",
        "model_name" : MODEL_NAME,
        "device"     : device,
        "timestamp"  : datetime.datetime.now().isoformat()
    },
    "setup": {
        "pytorch_version" : torch.__version__,
        "numpy_version"   : np.__version__,
        "pandas_version"  : pd.__version__
    },
    "architecture": {
        "embedding_dimension"    : int(embedding_dim),
        "num_hidden_layers"      : int(num_hidden_layers) if num_hidden_layers else None,
        "hidden_size"            : int(hidden_size) if hidden_size else None,
        "num_attention_heads"    : int(num_attention_heads) if num_attention_heads else None,
        "vocab_size"             : int(vocab_size) if vocab_size else None,
        "max_position_embeddings": int(max_position_emb) if max_position_emb else None,
        "intermediate_size"      : int(intermediate_size) if intermediate_size else None,
        "total_params"           : int(total_params),
        "trainable_params"       : int(trainable_params)
    },
    "tokenizer": {
        "type"            : type(hf_tokenizer).__name__,
        "vocab_size"      : int(hf_tokenizer.vocab_size),
        "model_max_length": int(hf_tokenizer.model_max_length),
        "cls_token"       : hf_tokenizer.cls_token,
        "sep_token"       : hf_tokenizer.sep_token,
        "pad_token"       : hf_tokenizer.pad_token,
        "unk_token"       : hf_tokenizer.unk_token,
        "special_tokens"  : hf_tokenizer.all_special_tokens
    },
    "pooling_and_normalization": {
        "pooling_strategy"         : "mean_pooling",
        "normalization"            : "L2 (normalize_embeddings=True)",
        "norm_l2_with_normalize"   : round(norm_st_norm, 6),
        "norm_l2_without_normalize": round(norm_st_raw, 6),
        "manual_vs_st_max_diff"    : round(diff, 10)
    },
    "truncation": {
        "max_seq_length" : int(max_seq),
        "tests"          : truncation_records
    },
    "embeddings_eja": {
        "n_sentences"  : len(eja_sentences),
        "embedding_dim": int(embeddings.shape[1]),
        "global_min"   : round(float(embeddings.min()), 6),
        "global_max"   : round(float(embeddings.max()), 6),
        "global_mean"  : round(float(embeddings.mean()), 6),
        "global_std"   : round(float(embeddings.std()), 6),
        "per_sentence" : stats
    }
}

# Salva o relatório
report_path = metrics_dir / "01_inspection_report.json"
with open(report_path, "w", encoding="utf-8") as f:
    json.dump(inspection_report, f, indent=2, ensure_ascii=False)

print(f"Relatório salvo em: {report_path.resolve()}")

# ── Tabela resumo ─────────────────────────────────────────────────────────────
summary_rows = [
    ("Modelo",                   MODEL_NAME),
    ("Dispositivo",              device),
    ("Dimensão do embedding",    embedding_dim),
    ("Camadas ocultas",          num_hidden_layers),
    ("Tamanho oculto",           hidden_size),
    ("Cabeças de atenção",       num_attention_heads),
    ("Tamanho do vocabulário",   f"{vocab_size:,}"),
    ("Total de parâmetros",      f"{total_params:,}"),
    ("Comprimento máximo (tokens)", max_seq),
    ("Estratégia de pooling",    "Mean Pooling"),
    ("Normalização",             "L2 (norma = 1.0)"),
    ("Norma L2 (com normaliz.)", f"{norm_st_norm:.6f}"),
    ("Dif. máx. manual vs ST",   f"{diff:.2e}"),
    ("Textos truncados (testes)", sum(1 for r in truncation_records if r['truncado'])),
    ("Plot salvo em",            str(plot_path))
]

df_summary = pd.DataFrame(summary_rows, columns=["Característica", "Valor"])
print("\n" + "=" * 70)
print("TABELA RESUMO DA INSPEÇÃO TÉCNICA")
print("=" * 70)
print(df_summary.to_string(index=False))

---
## Conclusões e Observações

### Arquitetura
- O **all-MiniLM-L12-v2** é um modelo BERT compacto com **12 camadas**, **384 dimensões** de embedding
  e **12 cabeças de atenção** — significativamente menor que o BERT-base (768 dim, 12 camadas),
  mas com excelente relação custo-benefício em tarefas de similaridade semântica.
- Conta com aproximadamente **33 milhões de parâmetros**, tornando-o adequado para ambientes
  com recursos computacionais limitados.

### Tokenizer
- Utiliza **WordPiece** (mesmo do BERT), com vocabulário de **~30.000 tokens**.
- O limite prático de **256 tokens** (já incluindo `[CLS]` e `[SEP]`) é suficiente para a
  maioria das perguntas e respostas curtas em contexto EJA, mas textos longos sofrerão truncamento.

### Pooling e Normalização
- O **mean pooling** ponderado pela `attention_mask` produz representações estáveis.
- A normalização L2 garante **norma unitária**, tornando o produto escalar equivalente à
  similaridade de cosseno — fundamental para recuperação eficiente por similaridade.
- A implementação manual confirmou equivalência com a saída nativa do SentenceTransformer.

### Truncamento
- Textos com mais de **256 tokens** são truncados silenciosamente, podendo causar perda de
  contexto. Recomenda-se pré-processar ou segmentar textos longos antes da codificação.

### Próximos Passos
- **Notebook 02:** Avaliação de similaridade semântica em pares de perguntas EJA.
- **Notebook 03:** Benchmarks de velocidade (latência de inferência, throughput por batch).
- **Notebook 04:** Avaliação em datasets de recuperação de informação (MRR, NDCG, MAP).

---
*Gerado automaticamente — Projeto Residência | Avaliação do Modelo all-MiniLM-L12-v2*